# TKCE — shallow trees + strong regularization

Combines every anti-overfitting lever at once:
- **shallow RF** (depth sweep 3, 4; `--rf-min-leaf 20`) -> coarser encoding
- **dropout 0.4**, **L1 = 5e-5**, **weight-decay 1e-3**
- **batch size 32**, **lr 3e-4**

### How to run
1. **Runtime -> Change runtime type -> GPU** (A100 on Pro+).
2. **Runtime -> Run all.**
3. When cell **4** asks, upload `openml_cache_361070.tar.gz` from your Desktop (OpenML API is down).

> **Heads-up: ~1-1.5 hours** (batch 32 is ~4x slower; 2 depths x 4 ablation models x 800 epochs). Keep the tab awake so Colab doesn't disconnect. To go faster, change `[3, 4]` to `[3]` in cell 7.

**What to look for** in each `x + tree` block: does `train_auc` stop hitting 1.0? does the train-test gap shrink? does test AUC hold ~0.69 or climb toward the tree's 0.708? (Compare against your earlier depth-6, no-reg run where x+tree=0.693.)

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4 · upload the dataset cache bundle (OpenML API is down)
from google.colab import files
print('Upload openml_cache_361070.tar.gz from your Desktop:')
files.upload()

In [ ]:
# 5 · extract the cache
import os, glob, tarfile
hits = glob.glob('/content/**/openml_cache_361070.tar.gz', recursive=True)
assert hits, 'Upload the bundle in cell 4 first.'
dst = '/root/.cache/openml/org/openml/www'
os.makedirs(dst, exist_ok=True)
with tarfile.open(hits[0]) as t:
    t.extractall(dst)
print('extracted from', hits[0])
print('tasks:', os.listdir(dst+'/tasks'), '| datasets:', os.listdir(dst+'/datasets'))

In [ ]:
# 6 · verify the dataset loads offline
import openml
t = openml.tasks.get_task(361070, download_splits=False)
d = t.get_dataset(); X, y, *_ = d.get_data(target=d.default_target_attribute)
print('OK cached, no network:', X.shape)

In [ ]:
# 7 · shallow RF + strong regularization  (sweep RF depth = 3, 4)
# levers: rf-depth low, rf-min-leaf 20, dropout 0.4, l1 5e-5, batch 32, lr 3e-4
for d in [3, 4]:
    print(f'\n########  RF depth={d} | dropout 0.4 | l1 5e-5 | batch 32 | lr 3e-4  ########\n')
    !python -u run_fusion.py --task 361070 --epochs 800 --fusion tabresnet --rf-depth {d} --rf-min-leaf 20 --batch-size 32 --dropout 0.4 --l1 5e-5 --weight-decay 1e-3 --lr 3e-4 --out results/fusion/depth_{d} --device auto --ablation

In [ ]:
# 8 · show the figures for each RF depth
from IPython.display import Image, display
import os
for d in [3, 4]:
    curves = f'results/fusion/depth_{d}/fusion_eye_movements_curves.png'
    bar    = f'results/fusion/depth_{d}/fusion_eye_movements.png'
    if os.path.exists(bar):
        print(f'==================  RF depth {d}  ==================')
        display(Image(bar)); display(Image(curves))

In [ ]:
# 9 · download everything as one zip
import shutil
from google.colab import files
shutil.make_archive('fusion_shallow_reg', 'zip', 'results/fusion')
files.download('fusion_shallow_reg.zip')